# 06 · ARM repair and cross-emulator confirmation

The complete CPU-emulation archive was generated by the included C firmware and study runner. This notebook verifies the archive and reruns a paired slice on Unicorn and Renode when Arm GCC and Renode are available. The test firmware checks every retained INT8 score against the dense oracle. Each emulator's instruction count is retained.

Set `TINYML_NOTEBOOK_REBUILD=1` to rebuild this paired slice. To reproduce the complete archive, run `python -m tinyml.study_v2 --workers 3`.

In [1]:
import json
import os
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "study.toml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tinyml.config import DATASETS, SEEDS
from tinyml.folds import CLIP_DIR, FOLDS, model_dir, read_fixture
from tinyml.study_v2 import V2

REBUILD = os.getenv("TINYML_NOTEBOOK_REBUILD") == "1"
WORKERS = os.getenv("TINYML_NOTEBOOK_WORKERS", "3")
DATA = Path(os.getenv("TINYML_DATA_DIR", ROOT / "data")).resolve()
SCRATCH = ROOT / "build" / "notebooks_v2"

def run(*args):
    proc = subprocess.run([sys.executable, *map(str, args)], cwd=ROOT, capture_output=True, text=True)
    print(proc.stdout + proc.stderr, end="")
    proc.check_returncode()

print("Saved-result verification" if not REBUILD else "Rebuild and verification")

Rebuild and verification


In [2]:
from tinyml.analysis_v2 import agreement_table, load_archive, strata, compare
from tinyml.counter_check import check

samples = load_archive()
display(samples.groupby(["cohort", "backend"]).agg(
    runs=("case_id", "size"), distinct_participants=("subject", "nunique"),
    score_pairs=("score_pairs", "sum"), mismatches=("mismatches", "sum"),
))
tables = SCRATCH / "agreement"
tables.mkdir(parents=True, exist_ok=True)
agreement, overlap = agreement_table(samples, tables)
display(agreement.round(3))

,,runs,distinct_participants,score_pairs,mismatches
cohort,backend,,,,
ablation,unicorn,20130,61,33568920,0
append0,unicorn,427,61,712068,0
append8,unicorn,854,61,1424136,0
cache16,unicorn,1830,61,3051720,0
cache4,unicorn,1830,61,3051720,0
cache8,unicorn,1830,61,3051720,0
horizon24,unicorn,1830,61,1557000,0
horizon8,unicorn,1830,61,560520,0
m33,unicorn,3660,61,6103440,0


,dataset,mode,matched,median_difference_pct,largest_difference_pct,median_append_difference_pct
0,wisdm,replay,255,0.066,1.980,0.114
1,wisdm,tap_equal,255,0.066,1.980,0.114
2,wisdm,recent_replay,255,0.069,2.899,0.114
3,wisdm,accumulator,255,0.590,2.899,0.114
4,mhealth,replay,50,0.063,0.990,0.114
5,mhealth,tap_equal,50,0.062,0.990,0.114
6,mhealth,recent_replay,50,0.066,1.449,0.114
7,mhealth,accumulator,50,0.731,1.449,0.114


In [3]:
if REBUILD:
    from tinyml.arm import run_arm
    from tinyml.cpu import build_runner, run_cpu
    from tinyml.protocol_v2 import METHODS, scenarios
    from tinyml.quant import Int8Model

    print("Renode instruction counter check:", check())
    target = SCRATCH / "cpu_smoke"
    runner = build_runner(target / "cpu_runner")
    rows = []
    names = ("replay", "tap_equal", "recent_replay", "accumulator")
    modes = tuple(METHODS[name] for name in names)
    for dataset in DATASETS:
        model = Int8Model.load(model_dir(V2, dataset, 0, SEEDS[0]) / "int8.npz")
        clip = read_fixture(dataset, 0)[1]
        cases = [case for case in scenarios(clip)
                 if case.placeholder == "hold"
                 and (case.pattern == "single" and case.delay in (1, 3, 7, 32)
                      or case.pattern == "burst4" and case.delay == 7)]
        a = run_arm(model, cases, 48, target / dataset / "renode", extended=True, mode_ids=modes)
        b = run_cpu(model, cases, 48, target / dataset / "unicorn", modes, runner)
        for renode, unicorn in zip(a, b, strict=True):
            for key in ("case_index", "mode_id", "mismatches", "stack_bytes", "state_bytes", "checksum"):
                assert renode[key] == unicorn[key]
            rows.append({
                "dataset": dataset, "scenario": cases[renode["case_index"]].case_id.split("/", 1)[1],
                "mode": names[modes.index(renode["mode_id"])],
                "Renode instructions": renode["instructions"],
                "Unicorn instructions": unicorn["instructions"],
            })
    display(pd.DataFrame(rows))
    print(f"{len(rows)} freshly emulated pairs agreed on every score check and SRAM measure.")

Renode instruction counter check: {'expected': 11, 'observed': 11, 'compiler': 'arm-none-eabi-gcc (15:14.2.rel1-1) 14.2.1 20241119', 'simulator': 'Renode v1.17.0'}


,dataset,scenario,mode,Renode instructions,Unicorn instructions
0,wisdm,single/1/hold,replay,36933,36906
1,wisdm,single/1/hold,tap_equal,38422,38393
2,wisdm,single/1/hold,recent_replay,38089,38059
3,wisdm,single/1/hold,accumulator,23029,22889
4,wisdm,single/3/hold,replay,73751,73698
5,wisdm,single/3/hold,tap_equal,57433,57390
6,wisdm,single/3/hold,recent_replay,57058,57014
7,wisdm,single/3/hold,accumulator,34445,34233
8,wisdm,single/7/hold,replay,147397,147302
9,wisdm,single/7/hold,tap_equal,108288,108224


40 freshly emulated pairs agreed on every score check and SRAM measure.
